# CSBS Caregiver REDCap Validation

Use this notebook to validate that REDCap CSBS Caregiver scoring formulas in the sandbox match the approved worksheet logic.

This notebook does not deploy changes. REDCap remains the implementation target.

In [ ]:
import os
import requests
import pandas as pd

API_URL = "https://redcap.research.sc.edu/api/"
TOKEN = os.getenv("NANO_API_TOKEN", "").strip()
if not TOKEN:
    raise RuntimeError("Set NANO_API_TOKEN in your environment before running.")

def redcap_post(content, **params):
    data = [("token", TOKEN), ("content", content), ("format", "json"), ("returnFormat", "json")]
    for key, value in params.items():
        if value is None:
            continue
        if isinstance(value, (list, tuple)):
            for i, item in enumerate(value):
                data.append((f"{key}[{i}]", str(item)))
        else:
            data.append((key, str(value)))
    resp = requests.post(API_URL, data=data, timeout=180)
    resp.raise_for_status()
    payload = resp.json()
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(payload["error"])
    return payload

meta = redcap_post("metadata")
csbs = [r for r in meta if r.get("form_name") == "csbs_caregiver"]
df = pd.DataFrame(csbs)
print("CSBS caregiver metadata rows:", len(df))
df.head(3)

In [ ]:
targets = [
    "csbs_emotionandeyegaze", "csbs_communication", "csbs_gestures",
    "csbs_sounds", "csbs_words", "csbs_understanding", "csbs_objectuse",
    "csbs_socialcomposite", "csbs_speechcomposite", "csbs_symboliccomposite",
    "cbscg_totalscore", "csbs_totalss", "csbs_totalss_auto", "csbs_totalper",
]

out = df[df["field_name"].isin(targets)][["field_name", "field_label", "field_type", "select_choices_or_calculations", "branching_logic", "field_annotation"]].copy()
out = out.sort_values("field_name").reset_index(drop=True)
pd.set_option("display.max_colwidth", 180)
out

In [ ]:
expected = {
    "csbs_socialcomposite": "[csbs_emotionandeyegaze]+[csbs_communication]+[csbs_gestures]",
    "csbs_speechcomposite": "[csbs_sounds]+[csbs_words]",
    "csbs_symboliccomposite": "[csbs_understanding]+[csbs_objectuse]",
    "cbscg_totalscore": "[csbs_socialcomposite]+[csbs_speechcomposite]+[csbs_symboliccomposite]",
}

check = out[out["field_name"].isin(expected.keys())][["field_name", "select_choices_or_calculations"]].copy()
check["expected"] = check["field_name"].map(expected)
check["match"] = check["select_choices_or_calculations"].str.replace(" ", "", regex=False) == check["expected"].str.replace(" ", "", regex=False)
check

In [ ]:
auto_row = out[out["field_name"] == "csbs_totalss_auto"].copy()
if auto_row.empty:
    raise RuntimeError("csbs_totalss_auto was not found in metadata.")

formula = auto_row.iloc[0]["select_choices_or_calculations"] or ""
branch = auto_row.iloc[0]["branching_logic"] or ""
print("Branching logic:", branch)
print("Contains 24-month event guard in calc:", "24_months_arm_1" in formula)
print("Contains lower-bound guard:", "<0" in formula or "< 0" in formula)
print("Contains upper-bound guard:", ">139" in formula or "> 139" in formula)
print("Starts with:", formula[:160])
print("Ends with:", formula[-160:])